# Oppitunti 18: AI-agenttien suojaaminen kryptografisilla kuiteilla

## Käytännön muistikirja

Tässä muistikirjassa käydään läpi neljä harjoitusta:

1. **Allekirjoita ensimmäinen kuitisi** agenttityökalun kutsulle ja varmista se.
2. **Väärennä kuittia** ja katso, että varmistus epäonnistuu.
3. **Rakenna kolmesta kuitista ketju** ja vahvista ketjun eheys.
4. **Kääri Microsoft Agent Framework -työkalukutsu** niin, että jokainen toiminto tuottaa kuitin.

Kaikki kryptografiset perusosat tuodaan hyvin ylläpidetyistä kirjastoista (`pynacl` Ed25519:lle, `jcs` RFC 8785 kanoniseen JSON:iin, `hashlib` Pythonin standardikirjastosta SHA-256:lle). Kuittien logiikka itsessään on puhdasta Pythonia, jota voit lukea ja muokata.

Suorita solut järjestyksessä. Jokainen osio on lyhyt ja itsenäinen.


## Asennus

Asenna nämä kaksi riippuvuutta. Molemmilla on sallivat lisenssit (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Apuohjelmat

Nämä kaksi apuohjelmaa käsittelevät base64url-koodausta (ilman täyteosia) ja mielivaltaisten objektien SHA-256-tiivistämistä. Ne pitävät muuhun muistiinpanoon keskittyvän logiikan itse kuittiin.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Allekirjoita ensimmäinen kuittisi

Kuvittele, että agenttimme **Contoso Travel** -yhtiössä juuri etsi lentoja Sydneystä Los Angelesiin asiakkaalle. Haluamme tallentaa tämän työkalukutsun allekirjoitettuna kuitiksi, jotta tuleva tarkastaja voi varmistaa sen ilman, että hänen tarvitsee luottaa meihin.

### Vaihe 1.1: Luo allekirjoitusavain

Tuotannossa agentin allekirjoitusavain sijaitsee laitteistoturva-moduulissa (HSM), Azure Key Vaultissa tai vastaavassa suojatussa säilytyspaikassa. Tässä oppitunnissa luomme uuden avaimen muistissa.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Vaihe 1.2: Rakenna kuittauksen tietopaketti

Tietopaketti sisältää kaiken, mitä haluamme kuitata: kuka toimi, mitä työkalua käytti, millä argumenteilla, mitä palautui, minkä politiikan alaisena ja milloin. Haluamme hajauttaa argumentit ja tuloksen sen sijaan, että sisällytämme ne suoraan, jotta kuittaus ei vuoda arkaluontoista sisältöä.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Vaihe 1.3: Allekirjoita ja kokoa kuitti

Kolme vaihetta:

1. Canonicalisoi sisältö JCS:n avulla, jotta kaksi toteutusta, jotka tuottavat saman loogisen kuitin, tuottavat tavut täsmälleen identtisinä.
2. Tee kanonisten tavujen hajautus SHA-256:lla.
3. Allekirjoita hajautusarvo Ed25519-yksityisavaimella.

Allekirjoitus liitetään sitten alkuperäiseen sisältöön tuottamaan lopullinen kuitti.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Vaihe 1.4: Vahvista kuitti

Vahvistus kääntää prosessin päinvastaiseksi. Poistamme allekirjoituksen, laskemme kannon hashin uudelleen ja tarkistamme allekirjoituksen kuitissa olevaa julkista avainta vastaan.

Tarkastajalta, joka tekee tämän vahvistuksen, ei tarvita meiltä muuta kuin itse kuitti. Ei tarvitse soittaa palvelulle, tietää avainhakemistosta tai luottaa mihinkään.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Sinun pitäisi nähdä `Receipt is valid: True`. Agentti on tuottanut ensimmäisen kryptografisesti allekirjoitetun tarkastustietueensa.


## Osa 2: Muokkaa kuittia

Koko kuittien tarkoitus on, että niiden käsittely on havaittavissa. Todistetaan se.

Muokkaamme yhtä merkkiä kuitissa ja katsomme, että tarkistus epäonnistuu.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Mitä juuri tapahtui?

Kun muutimme `policy_id`:tä, kanoniset tavut muuttuivat. Näiden tavujen SHA-256-tiiviste muuttui. Allekirjoitus (joka oli alkuperäisen tiivisteen yli) ei enää vastaa uutta tiivistettä. Varmennus palauttaa oikein `False`.

Ei ole mahdollista muuttaa mitään kuittitietuetta siten, että se silti varmennettaisiin, ellei hyökkääjällä ole yksityistä avainta. Niin kauan kuin yksityinen avain on avainvarastossa ja julkinen avain on julkaistu, väärentäminen on mahdotonta peittää.

Kokeile itse: muuta edellä olevan solun `tool_name` tai `agent_id` tai `timestamp` ja suorita uudelleen. Jokainen muutos tuottaa virheellisen kuitin.


## Osa 3: Ketjuta kuitit yhteen

Yksi kuitti suojaa yhtä toimintoa. Useimmat agentit suorittavat useita toimintoja. Jotta koko sekvenssi olisi väärentämisen kannalta havaittavissa, yhdistämme jokaisen kuitin edelliseen sisällyttämällä edellisen kuitin tiiviste uuden kuitin lataukseen.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Jos joku poistaa tai muuttaa kuitin järjestystä, ketju katkeaa juuri siinä kohdassa. Minkä tahansa myöhemmän kuitin vahvistus epäonnistuu, koska sen `previous_receipt_hash` ei enää vastaa edeltäjänsä todellista tiivistettä.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Riko ketju muuttamalla keskimmäistä kuittia ja tarkista uudelleen. Muutettu kuitti epäonnistuu allekirjoituksen tarkistuksessa, JA seuraava kuitti epäonnistuu ketjun linkin tarkistuksessa (koska sen `previous_receipt_hash` ei enää vastaa muokatun keskimmäisen kuitin hajautusarvoa).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Kuitti 0 vahvistuu edelleen (sitä ei muokattu eikä sillä ole edeltäjää, johon se perustuisi). Kuitti 1 epäonnistuu allekirjoituksen tarkistuksessa, koska muokkasimme `tool_args_hash`-arvoa. Kuitti 2 epäonnistuu ketjun linkin tarkistuksessa, koska sen `previous_receipt_hash` laskettiin alkuperäisen (nyt muokatun) kuitti 1:n perusteella.

Vaikka hyökkääjä allekirjoittaisi uudelleen muokatun kuitti 1:n (mikä ei ole mahdollista ilman yksityisavainta), ketjun linkin epäyhtenäisyys kuitissa 2 paljastaisi silti manipuloinnin. Muutoksen piilottamiseksi hyökkääjän pitäisi allekirjoittaa uudelleen jokainen kuitti muutospisteestä eteenpäin, mikä vaatii yksityisavaimen hallussapitoa.


## Osa 4: Kääri agenttityökalun kutsu kuittauksen allekirjoituksella

Todellisessa käyttöönotossa et halua, että jokainen agentin tekijä muistaa kutsua `make_receipt`. Haluat, että kuittauksen allekirjoitus on automaattinen jokaiselle työkalun kutsulle.

Tässä on yksinkertaisin malli: kääreluokka, joka ottaa minkä tahansa kutsuttavan työkalufunktion ja palauttaa siitä kuitin tuottavan version. Tämä mukautuu mihin tahansa agenttikehykseen, mukaan lukien Microsoft Agent Framework (`agent_framework.azure`).

Jos sinulla ei ole Azure AI Foundry -projektia käytössä, alla oleva paikallinen simulaatio demonstroi silti mallia.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Integrointi Microsoft Agent Frameworkin kanssa

Yllä oleva `ReceiptedTool`-kääre on kehykseen sitoutumaton. Käyttääksesi sitä agentin sisällä, joka on rakennettu Microsoft Agent Frameworkilla, rekisteröi kääritty funktio työkaluna. Luonnos (korvaat mockin oikealla Azure AI Foundry -työkalun rekisteröinnillä):

```python
# Pseudokoodi, joka näyttää integraation muodon.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     ohjeet="Olet Contoso Travelin agentti ...",
#     työkalut=[receipted_lookup],   # kääritty työkalu, ei raakaa funktiota
# )
# vastaus = agent.run("Etsi lennot Sydneystä Los Angelesiin kesäkuussa.")
#
# # Suorituksen jälkeen jokaisella agentin tekemällä työkalukutsulla on allekirjoitettu kuitti:
# audit_chain = receipted_lookup.receipts
```

Agenttikehyksen ei tarvitse tietää kuitista mitään. Kuitin allekirjoitus on kääritty työkalun ympärille, ei kiinnitetty kehykseen. Näin lisäät jäljitettävyyden olemassa olevaan agenttikoodiin kirjoittamatta agenttia uudelleen.


## Kertaus ja haastetehtävä

Olet:

- Generoinut Ed25519-avainparin.
- Luonut ja allekirjoittanut kuittauksen agenttityökalukutsulle.
- Vahvistanut kuitin offline-tilassa käyttäen vain julkista avainta.
- Muokannut kuittauksen sisältöä ja havainnut vahvistuksen epäonnistuvan.
- Rakentanut hajautetun ketjun kolmesta kuitista.
- Muokannut ketjun keskellä olevaa kuittia ja havainnut sekä allekirjoituksen epäonnistumisen että ketjuyhteyden katkeamisen.
- Kääriyt työkalufunktion automaattiseen kuittauksen allekirjoittamiseen.

**Haastetehtävä.** Laajenna kuittauskaaviota `request_id`-kentällä (UUID hajautettuun jäljitykseen). Päivitä `make_receipt` sisällyttämään tämä kenttä ja varmista, että kuitit edelleen vahvistuvat kokonaisuudessaan. Muuta sitten kenttää allekirjoituksen jälkeen ja varmista, että vahvistus epäonnistuu. Tämä pakottaa sinut ymmärtämään, miten jokainen tavu kanonisessa koodauksessa vaikuttaa allekirjoitukseen.

**Tärkeä raja.** Kuitit todistavat kolme asiaa ja vain kolme asiaa: omistajuuden (tällä avaimella allekirjoitettiin tämä sisältö), eheyden (sisältö ei ole muuttunut allekirjoituksen jälkeen) ja järjestyksen (tämä kuitti on myöhempi kuin toinen kuitti). Ne eivät todista, että agentin toiminta oli oikea, että `policy_id`-kentässä nimetty sääntö oli todella arvioitu tai että agentti noudatti jokaista sääntöä. Kuitit ovat pohja. Hallinto on järjestelmä, jonka rakennat niiden päälle.

Lue opetuksen README uudelleen tuon rajan huomioiden. Yleisin virhe, jonka tiimit tekevät kuittien kanssa, on olettaa, että "meillä on kuitit" tarkoittaa "meitä hallitaan." Näin ei ole. Kuitit tekevät agenttien toiminnan auditoitavaksi. Ne eivät tee siitä oikeaa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
